# Musahhih F2-P1 / F3-P1: guarded QLoRA workflow

This Kaggle notebook prepares one of two matched training arms: **F2-P1** (2,000 Tibyan-derived records) or **F3-P1** (1,000 frozen QALB natural records plus 1,000 frozen Tibyan-derived records). It reuses the frozen F1-P1 model, prompt, LoRA, optimizer, development set, and checkpoint rule.

This committed notebook is **non-executing by default**. QALB test and Nahw-Passage are never loaded here. It does not generate predictions, evaluate a final test, or activate the proposed XG operator. Private records, adapters, checkpoints, logs, and model responses must remain in a private Kaggle notebook/dataset and outside Git and Notion. CPU is not a training fallback.

The methodology commit approved for implementation is `8ca3014e6b3659e2e8c3ffc519b0255e9af6b7a6`. A separate exact-commit GO on [issue #69](https://github.com/ALBA7OOTH-Research-Lab/Musahhih/issues/69) is required before either the GPU smoke or full training can run.

## 1. Choose one arm and leave execution disabled

Start with F2-P1. After its smoke and full run are preserved, restart from a fresh Kaggle runtime and change `ARM` to F3-P1. Do not run both arms in the same model process. The blank approval fields are deliberate safety gates.

In [ ]:
ARM = 'F2-P1'  # Allowed: 'F2-P1' or 'F3-P1'.
RUN_GPU_SMOKE = False
RUN_FULL_TRAINING = False
LOAD_PRIOR_SMOKE_SUMMARY = False
PRIOR_SMOKE_SUMMARY_PATH_VALUE = ''  # Private mounted file used after restart.
APPROVED_WORKFLOW_COMMIT = ''  # Fill only after exact-commit GO on issue #69.
APPROVAL_REFERENCE = ''  # Exact issue #69 comment URL containing that GO.
GPU_SMOKE_CONFIRMATION_VALUE = ''
FULL_TRAINING_CONFIRMATION_VALUE = ''
if RUN_GPU_SMOKE and RUN_FULL_TRAINING:
    raise RuntimeError('Smoke and full training require separate fresh runtimes.')
if RUN_GPU_SMOKE and LOAD_PRIOR_SMOKE_SUMMARY:
    raise RuntimeError('Choose a new smoke or a prior smoke summary, not both.')
EXECUTE_GPU_STAGE = RUN_GPU_SMOKE or RUN_FULL_TRAINING
print({'arm': ARM, 'gpu_smoke': RUN_GPU_SMOKE, 'full_training': RUN_FULL_TRAINING})

## 2. Safe repository setup

The cell clones the organization repository when absent. It fast-forwards an existing clean checkout and never deletes or overwrites local work.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
RUNTIME_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPO = RUNTIME_ROOT / 'Musahhih'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/ALBA7OOTH-Research-Lab/Musahhih.git', str(REPO)], check=True)
else:
    dirty = subprocess.run(['git', '-C', str(REPO), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout
    if dirty:
        print('Existing repository has local changes; leaving it unchanged.')
    else:
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
ACTUAL_WORKFLOW_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('Repository:', REPO)
print('Commit:', ACTUAL_WORKFLOW_COMMIT)

## 3. Validate the private training records without printing text

Attach the institutionally authorized **private** Kaggle dataset containing `f2_train_records.jsonl`, `f3_train_records.jsonl`, and `common_dev_records.jsonl`. The adapter can be rebuilt locally with `scripts/prepare_f2_f3_training_records.py`. This preflight checks exact hashes, schema, counts, roles, and aggregate provenance; it never prints corpus text. Keep the Kaggle notebook private.

In [ ]:
from scripts.f2_f3_training_utils import *
validate_arm(ARM)
if EXECUTE_GPU_STAGE:
    require_execution_approval(APPROVED_WORKFLOW_COMMIT, ACTUAL_WORKFLOW_COMMIT, APPROVAL_REFERENCE)
KAGGLE_PRIVATE_INPUT_DIR = Path('/kaggle/input/musahhih-f2-f3-private-records')
PRIVATE_INPUT_ROOT = KAGGLE_PRIVATE_INPUT_DIR if KAGGLE_PRIVATE_INPUT_DIR.exists() else Path('data/processed/f2_f3_training_records')
def locate_private_file(name):
    matches = sorted(PRIVATE_INPUT_ROOT.rglob(name)) if PRIVATE_INPUT_ROOT.exists() else []
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one private {name}; found {len(matches)} under {PRIVATE_INPUT_ROOT}')
    return matches[0]
TRAIN_PATH = locate_private_file('f2_train_records.jsonl' if ARM == 'F2-P1' else 'f3_train_records.jsonl')
DEV_PATH = locate_private_file('common_dev_records.jsonl')
PRIVATE_INPUT_SUMMARY = {
    'train': validate_private_records(TRAIN_PATH, ARM),
    'development': validate_private_records(DEV_PATH, 'development'),
}
print(json.dumps(PRIVATE_INPUT_SUMMARY, indent=2))
print('Only aggregate metadata was printed; private Arabic text remains private.')

## 4. GPU and dependency preflight (only when an execution flag is enabled)

The validated free runtime is one NVIDIA P100. In Kaggle, open **Settings → Accelerator → GPU P100**. GPU availability is not guaranteed. The cell fails before model loading if CUDA, exactly one visible GPU, or P100 is missing. It uses the same pinned compatibility stack as F1-P1 and disables Unsloth compilation on P100 because its CUDA capability is 6.0.

In [ ]:
RUNTIME = {'executed': False}
if EXECUTE_GPU_STAGE:
    assigned_gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip()
    if len([line for line in assigned_gpu_name.splitlines() if line.strip()]) != 1 or 'P100' not in assigned_gpu_name:
        raise RuntimeError('This frozen workflow requires exactly one Kaggle NVIDIA P100; CPU is not a training fallback.')
    import PIL
    P100_STACK = {'torch': '2.6.0', 'torchvision': '0.21.0', 'xformers': '0.0.29.post3', 'torchao': '0.16.0', 'numpy': '2.0.2', 'pillow': PIL.__version__, 'index': 'https://download.pytorch.org/whl/cu124'}
    os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall', f"torch=={P100_STACK['torch']}", f"torchvision=={P100_STACK['torchvision']}", f"xformers=={P100_STACK['xformers']}", f"numpy=={P100_STACK['numpy']}", f"Pillow=={P100_STACK['pillow']}", '--index-url', P100_STACK['index']], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', f"torchao=={P100_STACK['torchao']}"], check=True)
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable. Enable a Kaggle P100 and restart the session.')
    gpu = torch.cuda.get_device_properties(0)
    if torch.cuda.device_count() != 1:
        raise RuntimeError('Exactly one visible GPU is required.')
    torch.ones(1, device='cuda').sum().item()
    torch.cuda.synchronize()
    print('GPU:', gpu.name, 'VRAM GiB:', round(gpu.total_memory / 1024**3, 2), 'free GiB:', round(torch.cuda.mem_get_info()[0] / 1024**3, 2))
else:
    print('GPU stage disabled: CUDA and package installation were not touched.')

In [ ]:
if EXECUTE_GPU_STAGE:
    import re
    torch_minor = re.match(r'[0-9]+\.[0-9]+', torch.__version__).group(0)
    xformers_version = {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2','2.6':'0.0.29.post3'}.get(torch_minor, '0.0.34')
    def pip_install(*items):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *items], check=True)
    pip_install('sentencepiece', 'protobuf', 'datasets==4.3.0', 'huggingface_hub>=0.34.0', 'hf_transfer')
    pip_install('--no-deps', 'unsloth_zoo', 'bitsandbytes', 'accelerate', f'xformers=={xformers_version}', 'peft', 'trl', 'triton', 'unsloth')
    pip_install('--no-deps', '--upgrade', 'torchao==0.16.0')
    pip_install('transformers==4.56.2')
    pip_install('--no-deps', 'trl==0.22.2')
    import importlib.metadata as md, platform
    packages = ['torch','transformers','unsloth','accelerate','peft','trl','datasets','bitsandbytes']
    VERSIONS = {name: md.version(name) for name in packages}
    RUNTIME = {'executed': True, 'python': platform.python_version(), 'cuda': torch.version.cuda, 'gpu': gpu.name, 'cuda_capability': f'{gpu.major}.{gpu.minor}', 'visible_gpu_count': torch.cuda.device_count(), 'selected_device': 0, 'p100_stack': P100_STACK, 'unsloth_compile_disabled': os.environ.get('UNSLOTH_COMPILE_DISABLE') == '1', 'packages': VERSIONS}
    print(json.dumps(RUNTIME, indent=2))

## 5. Load the untouched base and attach a fresh LoRA

This stage runs only after an execution flag is enabled. It loads the exact frozen 4-bit Gemma checkpoint and attaches a new unmerged LoRA. It does not reuse F1 weights, attach an existing adapter, merge weights, or generate text.

In [ ]:
if EXECUTE_GPU_STAGE:
    from unsloth import FastModel
    from unsloth.chat_templates import get_chat_template
    model, processor = FastModel.from_pretrained(model_name=MODEL_ID, revision=MODEL_REVISION, max_seq_length=MAX_SEQUENCE_LENGTH, load_in_4bit=True)
    processor = get_chat_template(processor, chat_template='gemma-3')
    model = FastModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0.0, bias='none', target_modules=list(LORA_TARGETS), use_gradient_checkpointing='unsloth', random_state=SEED)
    MODEL_INFO = {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'max_sequence_length': MAX_SEQUENCE_LENGTH, 'load_in_4bit': True, 'fresh_lora': True, 'lora_targets': LORA_TARGETS}
    print(json.dumps(MODEL_INFO, indent=2))

## 6. Exact length guard and assistant-only trainer

The full Gemma chat template is measured for every private training record. Any sequence over 1,024 tokens stops the run; records are never truncated, replaced, or filtered. Loss is applied only to assistant correction tokens. The common frozen 975-record QALB development view is used only for epoch-level loss and checkpoint selection.

In [ ]:
if EXECUTE_GPU_STAGE:
    from datasets import load_dataset
    from trl import SFTTrainer, SFTConfig
    from unsloth.trainer import UnslothVisionDataCollator
    private_data = load_dataset('json', data_files={'train': str(TRAIN_PATH), 'validation': str(DEV_PATH)})
    def format_row(row):
        return {'messages': row['prompt'] + row['completion']}
    private_data = private_data.map(format_row, remove_columns=private_data['train'].column_names)
    text_tokenizer = getattr(processor, 'tokenizer', processor)
    def rendered_token_count(messages):
        rendered = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        input_ids = text_tokenizer(rendered, add_special_tokens=False, return_attention_mask=False)['input_ids']
        if input_ids and isinstance(input_ids[0], (list, tuple)):
            raise RuntimeError('Token-length guard received batched IDs.')
        return len(input_ids)
    lengths = [rendered_token_count(messages) for messages in private_data['train']['messages']]
    if not lengths or min(lengths) < 2 or max(lengths) > MAX_SEQUENCE_LENGTH:
        raise RuntimeError(f'Frozen sequence-length gate failed: min={min(lengths)}, max={max(lengths)}')
    LONGEST_INDEX = max(range(len(lengths)), key=lengths.__getitem__)
    OUTPUT_DIR = Path('outputs') / run_id(ARM, 'training')
    if OUTPUT_DIR.exists():
        raise RuntimeError('Run directory already exists; experiment IDs are never overwritten.')
    args = SFTConfig(output_dir=str(OUTPUT_DIR), max_length=MAX_SEQUENCE_LENGTH, dataset_text_field='', dataset_kwargs={'skip_prepare_dataset': True}, remove_unused_columns=False, completion_only_loss=True, eval_strategy='epoch', save_strategy='epoch', save_total_limit=2, report_to='none', bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(), seed=SEED, **TRAINING_CONFIG)
    response_collator = UnslothVisionDataCollator(model, processor, max_seq_length=MAX_SEQUENCE_LENGTH, train_on_responses_only=True, instruction_part='<start_of_turn>user\n', response_part='<start_of_turn>model\n', completion_only_loss=True)
    trainer = SFTTrainer(model=model, processing_class=processor, data_collator=response_collator, train_dataset=private_data['train'], eval_dataset=private_data['validation'], args=args)
    first_labels = response_collator([private_data['train'][0]])['labels'][0]
    if not torch.any(first_labels != -100).item():
        raise RuntimeError('Completion masking failed: no trainable assistant tokens.')
    LENGTH_SUMMARY = {'records': len(lengths), 'minimum': min(lengths), 'maximum': max(lengths), 'longest_index': LONGEST_INDEX, 'validated_sequence_lengths': True, 'contains_corpus_text': False}
    print(json.dumps(LENGTH_SUMMARY, indent=2))

## 7. Deliberate one-step longest-record GPU smoke

Do not enable this until issue #69 contains `GO for commit <exact notebook commit>`. Copy that 40-character commit and comment URL into section 1, set the exact confirmation to `RUN_F2_F3_LONGEST_RECORD_SMOKE`, and enable only `RUN_GPU_SMOKE`. The smoke performs one optimizer step on the longest record and requires at least 1 GiB headroom. That model process is then tainted: save the aggregate summary and restart before full training.

In [ ]:
SMOKE_RUNTIME_TAINTED = False
PRIVATE_ARTIFACT_ROOT = Path('/kaggle/working/musahhih-private-artifacts') if Path('/kaggle/working').exists() else Path('outputs')
SMOKE_SUMMARY_PATH = PRIVATE_ARTIFACT_ROOT / f'{run_id(ARM, "smoke")}_summary.json'
SMOKE_SUMMARY = {'passed': False, 'executed': False, 'arm': ARM, 'contains_corpus_text': False}
if LOAD_PRIOR_SMOKE_SUMMARY:
    if not PRIOR_SMOKE_SUMMARY_PATH_VALUE:
        raise RuntimeError('Set the private mounted prior smoke-summary path.')
    prior_smoke_path = Path(PRIOR_SMOKE_SUMMARY_PATH_VALUE)
    SMOKE_SUMMARY = json.loads(prior_smoke_path.read_text(encoding='utf-8'))
if RUN_GPU_SMOKE:
    require_smoke_confirmation(GPU_SMOKE_CONFIRMATION_VALUE, APPROVED_WORKFLOW_COMMIT, ACTUAL_WORKFLOW_COMMIT, APPROVAL_REFERENCE)
    if SMOKE_SUMMARY_PATH.exists():
        raise RuntimeError('Smoke summary already exists; never overwrite a run artifact.')
    torch.cuda.reset_peak_memory_stats()
    original_dataset, original_max_steps, original_output = trainer.train_dataset, trainer.args.max_steps, trainer.args.output_dir
    trainer.train_dataset = original_dataset.select([LONGEST_INDEX])
    trainer.args.max_steps = 1
    trainer.args.output_dir = str(PRIVATE_ARTIFACT_ROOT / f'{run_id(ARM, "smoke")}_temporary')
    trainer.train()
    peak = torch.cuda.max_memory_reserved()
    SMOKE_SUMMARY = memory_gate(gpu.total_memory, peak) | {'executed': True, 'arm': ARM, 'workflow_commit': ACTUAL_WORKFLOW_COMMIT, 'approval_reference': APPROVAL_REFERENCE, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'selection': LENGTH_SUMMARY, 'runtime': RUNTIME, 'contains_corpus_text': False}
    trainer.train_dataset, trainer.args.max_steps, trainer.args.output_dir = original_dataset, original_max_steps, original_output
    PRIVATE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    SMOKE_SUMMARY_PATH.write_text(json.dumps(SMOKE_SUMMARY, indent=2) + '\n', encoding='utf-8')
    SMOKE_RUNTIME_TAINTED = True
    if not SMOKE_SUMMARY['passed']:
        raise RuntimeError('GPU smoke failed the frozen 1 GiB headroom gate.')
print(json.dumps(SMOKE_SUMMARY, indent=2))
if SMOKE_RUNTIME_TAINTED:
    print('Save the smoke summary, stop this session, and restart before full training.')

## 8. Optional two-epoch training — disabled

Full training needs a fresh runtime, the preserved passing smoke summary for the same arm and workflow commit, the same exact-commit GO, and the arm-specific confirmation (`RUN_F2_P1_TWO_EPOCH_TRAINING` or `RUN_F3_P1_TWO_EPOCH_TRAINING`). The selected checkpoint is the lower mean assistant-token development loss at epoch 1 or 2; differences within 1e-6 choose epoch 1. No generation or final-test evaluation occurs here.

In [ ]:
if RUN_FULL_TRAINING:
    require_full_training_confirmation(ARM, FULL_TRAINING_CONFIRMATION_VALUE, SMOKE_SUMMARY, APPROVED_WORKFLOW_COMMIT, ACTUAL_WORKFLOW_COMMIT, APPROVAL_REFERENCE)
    if SMOKE_RUNTIME_TAINTED:
        raise RuntimeError('A smoke-tainted model cannot run full training; restart and load the prior summary.')
    result = trainer.train()
    evaluations = [item for item in trainer.state.log_history if 'eval_loss' in item and 'epoch' in item]
    if len(evaluations) != 2:
        raise RuntimeError(f'Expected exactly two epoch evaluations, observed {len(evaluations)}')
    first, second = evaluations
    selected = first if first['eval_loss'] <= second['eval_loss'] + 1e-6 else second
    SELECTED_CHECKPOINT = OUTPUT_DIR / f"checkpoint-{int(selected['step'])}"
    if not SELECTED_CHECKPOINT.is_dir():
        raise RuntimeError('Selected epoch checkpoint is missing.')
    selection_summary = {'arm': ARM, 'workflow_commit': ACTUAL_WORKFLOW_COMMIT, 'approval_reference': APPROVAL_REFERENCE, 'rule': 'lowest common-development assistant-token loss; ties within 1e-6 choose epoch 1', 'evaluations': evaluations, 'selected_checkpoint': SELECTED_CHECKPOINT.name, 'contains_corpus_text': False, 'nahw_passage_used': False, 'qalb_test_used': False}
    (OUTPUT_DIR / 'checkpoint_selection.json').write_text(json.dumps(selection_summary, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(selection_summary, indent=2))
else:
    print('Full training is disabled. No optimizer step was started by this section.')

## 9. Preserve private artifacts

Use Kaggle **Save Version** only while the notebook and output remain private, or download the private artifact directory locally. Do not commit or paste private records, checkpoints, adapters, logs, or model responses into GitHub or Notion. Record the Kaggle run URL and artifact checksums in the private lab record. A later separately frozen protocol will govern selected-adapter evaluation; do not load Nahw-Passage or QALB test in this notebook.